# SQL — Intermediate, Session 03: Window over-partition %, RANK ties, EXCEPT, INTERSECT

Dataset: `datasets/chinook.db`. Third intermediate notebook.

Select the **"Python (SCLT tutor venv)"** kernel before running anything.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../../../datasets/chinook.db")

def run(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

---
## Q1 — Window function: percent of total (no PARTITION BY)

From `InvoiceLine` joined to `Track` (on `TrackId`) joined to `Genre` (on `GenreId`),
compute per-`Genre.Name` revenue as `SUM(InvoiceLine.UnitPrice * InvoiceLine.Quantity)`,
grouped by `Genre.Name`, in a CTE called `GenreSales` with columns `GenreName` and
`GenreRevenue`. Then, in the final query against `GenreSales`, add a `PctOfTotal` column
computed as `ROUND(100.0 * GenreRevenue / SUM(GenreRevenue) OVER (), 2)` — note this
`SUM() OVER ()` has no `PARTITION BY`, so it sums across *all* rows, giving each row's
share of the grand total. Return `GenreName`, `GenreRevenue`, `PctOfTotal`, ordered by
`GenreRevenue` descending, limit 10.

In [6]:
query = """
-- your SQL here
WITH GenreSales AS (
    SELECT
        Genre.Name AS GenreName,
        SUM(InvoiceLine.UnitPrice * InvoiceLine.Quantity) AS GenreRevenue
    FROM
        InvoiceLine
    JOIN
        Track ON InvoiceLine.TrackId = Track.TrackId
    JOIN
        Genre ON Track.GenreId = Genre.GenreId
    GROUP BY
        Genre.Name
    )
SELECT
    GenreName,
    GenreRevenue,
    SUM(GenreRevenue) OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS TotalRevenue
FROM
    GenreSales
GROUP BY
    GenreName, 
    GenreRevenue
ORDER BY
    GenreRevenue DESC
LIMIT 10
"""
run(query)

,GenreName,GenreRevenue,TotalRevenue
0,Rock,826.65,2328.6
1,Latin,382.14,2328.6
2,Metal,261.36,2328.6
3,Alternative & Punk,241.56,2328.6
4,TV Shows,93.53,2328.6
5,Jazz,79.20,2328.6
6,Blues,60.39,2328.6
7,Drama,57.71,2328.6
8,Classical,40.59,2328.6
9,R&B/Soul,40.59,2328.6


---
## Q2 — EXCEPT

Find customers who have made at least one purchase but have **never** bought a track
from the `'Opera'` genre. Left side: `SELECT CustomerId FROM Invoice`. Right side:
`SELECT Invoice.CustomerId` from `Invoice` joined to `InvoiceLine` (on `InvoiceId`)
joined to `Track` (on `TrackId`) joined to `Genre` (on `GenreId`), `WHERE Genre.Name =
'Opera'`. Combine the two with `EXCEPT` (left minus right). Order the final result by
`CustomerId`.

In [ ]:
query = """
-- your SQL here

"""
run(query)

---
## Q3 — RANK() within a partition

From `PlaylistTrack` joined to `Track` (on `TrackId`), for `PlaylistId = 1` only,
compute `DurationRank` as `RANK() OVER (PARTITION BY PlaylistTrack.PlaylistId ORDER BY
Track.Milliseconds DESC)`. Return `Track.Name` as `TrackName`, `Track.Milliseconds`,
`DurationRank` for rows where `DurationRank <= 5`, ordered by `DurationRank`.

In [ ]:
query = """
-- your SQL here
"""
run(query)

---
## Q4 — Multi-CTE with RANK ties

Chain two CTEs. First, `TrackRevenue` — from `InvoiceLine` joined to `Track` (on
`TrackId`), grouped by `Track.GenreId`, `Track.TrackId`, `Track.Name`, with a
`Revenue` column as `SUM(InvoiceLine.UnitPrice * InvoiceLine.Quantity)`. Second,
`RankedTracks` — select from `TrackRevenue` and add `RevenueRank` as `RANK() OVER
(PARTITION BY GenreId ORDER BY Revenue DESC)`. In the final query, join `RankedTracks`
to `Genre` on `GenreId`, filter to `RevenueRank = 1`, and return `Genre.Name` as
`GenreName`, `TrackName`, `Revenue`, ordered by `GenreName`, limit 10.

**Heads up:** many tracks are tied at the same price, so `RANK() = 1` can return
*multiple* rows for the same genre (that's correct `RANK()` behavior on ties — it's not
a bug if you see the same `GenreName` repeated).

In [ ]:
query = """
-- your SQL here
"""
run(query)

---
## Q5 — INTERSECT

Find artists who have tracks in **both** `'Rock'` and `'Alternative & Punk'` genres.
Build a CTE `RockAndAltArtists` that intersects two subqueries: each selects `DISTINCT
Album.ArtistId` from `Album` joined to `Track` (on `AlbumId`) joined to `Genre` (on
`GenreId`), one filtered `WHERE Genre.Name = 'Rock'`, the other `WHERE Genre.Name =
'Alternative & Punk'`, combined with `INTERSECT`. In the final query, join
`RockAndAltArtists` to `Artist` on `ArtistId`, returning `Artist.ArtistId`,
`Artist.Name`, ordered by `Artist.Name`.

In [ ]:
query = """
-- your SQL here
"""
run(query)

---
## Done?

Say "done" in chat once all 5 run and look right.